# Notebook 04 — SHAP Explainability & Feature Analysis
## Pearls AQI Predictor · Hyderabad, Pakistan

**Objective:** Explain the trained model's predictions using SHAP (SHapley Additive exPlanations).
This notebook answers **"Why did the model make this prediction?"** for every forecast.

**What we do:**
1. Load the best trained model from Notebook 03
2. Initialize SHAP TreeExplainer (for tree models) or KernelExplainer (fallback)
3. Generate per-prediction explanations
4. Visualize global feature importance
5. Show waterfall plots for individual predictions
6. Generate natural-language explanation summaries
7. Fallback to correlation analysis if SHAP is unavailable

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from models.explainer import ModelExplainer, LIMEExplainer, correlation_explanation
from feature_store.feature_builder import FeatureBuilder
from utils.config import get
from utils.storage import load_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 60)

In [ ]:
# Load best model
MODEL_DIR = Path('../models/artifacts')
model_files = sorted(MODEL_DIR.glob('best_model_*.pkl'), reverse=True)

if model_files:
    import joblib
    best_model = joblib.load(model_files[0])
    print(f'✅ Loaded model: {model_files[0].name}')
    print(f'   Model type: {type(best_model).__name__}')
    if hasattr(best_model, 'model') and best_model.model is not None:
        print(f'   Underlying: {type(best_model.model).__name__}')
else:
    print('⚠️ No saved model found. Will use correlation-based importance.')
    best_model = None

In [ ]:
# Load feature data
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    builder = FeatureBuilder(df)
    featured = builder.build_all()
    print(f'✅ Loaded {len(featured)} featured rows')
except FileNotFoundError:
    print('⚠️ No data found, creating synthetic for demo')
    np.random.seed(42)
    n = 300
    featured = pd.DataFrame({
        'aqi': np.clip(60 + np.random.randn(n) * 25, 0, 300),
        'temperature_2m': 25 + np.random.randn(n) * 8,
        'relative_humidity_2m': np.clip(50 + np.random.randn(n) * 15, 0, 100),
        'wind_speed_10m': np.abs(3 + np.random.randn(n) * 2),
        'precipitation': np.abs(np.random.randn(n)) * 3,
        'cloud_cover': np.random.uniform(0, 100, n),
        'aqi_lag_1': np.clip(60 + np.random.randn(n) * 25, 0, 300),
        'aqi_lag_24': np.clip(60 + np.random.randn(n) * 25, 0, 300),
        'aqi_roll_mean_24': np.clip(60 + np.random.randn(n) * 15, 0, 300),
    })

# Extract feature columns
feature_cols = [
    c for c in featured.columns
    if not c.startswith('target_')
    and c not in ('timestamp', 'source', 'station_name', 'city', 'country',
                  'dominant_pollutant', 'merged_at', 'fetched_at', 'latitude', 'longitude')
    and featured[c].dtype in (np.float64, np.float32, np.int64, np.int32)
]

print(f'Feature columns: {len(feature_cols)}')

---
## 1. SHAP TreeExplainer (Primary Method)

If the best model is tree-based (Random Forest, XGBoost, LightGBM, Gradient Boosting),
we use SHAP's TreeExplainer — it's fast and exact. For other model types, KernelExplainer is the fallback.

In [ ]:
X = featured[feature_cols].fillna(0).values

# Try SHAP
shap_available = False
if best_model is not None:
    # If the best model is a SklearnWrapper, extract the underlying sklearn model
    underlying_model = best_model.model if hasattr(best_model, 'model') else best_model

    explainer = ModelExplainer(underlying_model, feature_cols)
    shap_ok = explainer.fit_shap(X[:100])  # Use first 100 as background

    if shap_ok:
        shap_available = True
        print('✅ SHAP initialized successfully')
    else:
        print('⚠️ SHAP initialization failed — will use correlation fallback')
else:
    print('⚠️ No trained model — using correlation-based importance')

---
## 2. Global Feature Importance

Which features matter most **on average** across all predictions?

In [ ]:
if shap_available:
    # Get SHAP values for a sample
    sample_size = min(200, len(X))
    explanation = explainer.explain(X[-sample_size:])

    if explanation and explanation.get('top_drivers'):
        drivers = explanation['top_drivers'][:15]

        fig, ax = plt.subplots(figsize=(12, 7))
        names = [d['feature'].replace('_', ' ') for d in reversed(drivers)]
        values = [d.get('abs_importance', d.get('shap_value', 0)) for d in reversed(drivers)]
        directions = [d.get('direction', 'neutral') for d in reversed(drivers)]
        colors = ['#ff3333' if d == 'positive' else '#00e400' if d == 'negative' else '#9898b8' for d in directions]

        ax.barh(names, values, color=colors, alpha=0.8)
        ax.set_title('Global Feature Importance (Mean |SHAP|)', fontsize=14)
        ax.set_xlabel('Mean |SHAP Value|')
        ax.grid(True, alpha=0.15, axis='x')
        plt.tight_layout()
        plt.show()

        print('Method:', explanation.get('method', 'unknown'))
        print(f'\nTop 5 drivers:')
        for d in drivers[:5]:
            arrow = '↑' if d.get('direction') == 'positive' else '↓'
            print(f'  {arrow} {d["feature"]:<35} (impact: {d.get("abs_importance", d.get("shap_value", 0)):.4f})')

---
## 3. Individual Prediction Explanation

What drove **this specific prediction**?

In [ ]:
if shap_available:
    # Explain the latest prediction (last row of data)
    single_explanation = explainer.explain(X[-1:])

    if single_explanation and single_explanation.get('top_drivers'):
        drivers = single_explanation['top_drivers']

        # Waterfall-like visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        names = [d['feature'].replace('_', ' ') for d in drivers]
        values = [d.get('shap_value', d.get('importance', 0)) for d in drivers]
        colors = ['#ff3333' if v > 0 else '#00e400' for v in values]

        ax.barh(range(len(names)), values, color=colors, alpha=0.8)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.axvline(0, color='white', linewidth=0.5)
        ax.set_title('Feature Contributions — Latest Prediction', fontsize=14)
        ax.set_xlabel('SHAP Value (contribution to prediction)')
        ax.grid(True, alpha=0.15, axis='x')
        plt.tight_layout()
        plt.show()

        print('🔍 Natural-Language Explanation:')
        print(f'   {single_explanation.get("natural_language", "")}')

---
## 4. SHAP Summary Plot (Beeswarm)

Shows the distribution of SHAP values for each feature — reveals both magnitude and direction.

In [ ]:
if shap_available:
    try:
        import shap
        sample_idx = min(150, len(X))
        shap_vals = explainer.shap_explainer.shap_values(X[-sample_idx:])

        if isinstance(shap_vals, list):
            shap_vals = shap_vals[0]

        fig, ax = plt.subplots(figsize=(12, 8))
        shap.summary_plot(
            shap_vals,
            featured[feature_cols].iloc[-sample_idx:].fillna(0),
            feature_names=[f.replace('_', ' ') for f in feature_cols],
            plot_type='dot',
            show=False,
            max_display=15
        )
        plt.title('SHAP Summary Plot — Feature Impact Distribution', fontsize=14)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'SHAP summary plot failed: {e}')

---
## 5. Correlation-Based Importance (Fallback)

When SHAP is unavailable (no trained tree model), Pearson correlation with AQI provides a useful fallback.

In [ ]:
print('=== Correlation-Based Feature Importance (Always Available) ===\n')

corr_result = correlation_explanation(featured, 'aqi')

if corr_result.get('top_drivers'):
    drivers = corr_result['top_drivers'][:15]

    fig, ax = plt.subplots(figsize=(12, 7))
    names = [d['feature'].replace('_', ' ') for d in reversed(drivers)]
    values = [d.get('importance', 0) for d in reversed(drivers)]
    directions = [d.get('direction', 'neutral') for d in reversed(drivers)]
    colors = ['#ff3333' if d == 'positive' else '#00e400' for d in directions]

    ax.barh(names, values, color=colors, alpha=0.8)
    ax.set_title('Correlation-Based Feature Importance (|r| with AQI)', fontsize=14)
    ax.set_xlabel('|Pearson r|')
    ax.grid(True, alpha=0.15, axis='x')
    plt.tight_layout()
    plt.show()

    print('Method:', corr_result.get('method', 'correlation'))
    print(f'Natural language: {corr_result.get("natural_language", "")}')
    print(f'\nTop 5 correlated features:')
    for d in drivers[:5]:
        direction = 'positive' if d.get('direction') == 'positive' else 'negative'
        print(f'  {d["feature"]:<35} r={d.get("correlation", 0):.3f} ({direction})')
else:
    print('No correlation data available.')

---
## 6. Feature Interaction Deep Dive

Explore how key features interact with each other to drive AQI predictions.

In [ ]:
# Temperature vs Humidity colored by AQI
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

aqi_col = 'aqi' if 'aqi' in featured.columns else 'us_aqi'

# 1. Temperature vs Humidity colored by AQI
if 'temperature_2m' in featured.columns and 'relative_humidity_2m' in featured.columns:
    sample = featured[[aqi_col, 'temperature_2m', 'relative_humidity_2m']].dropna()
    sc = axes[0,0].scatter(sample['temperature_2m'], sample['relative_humidity_2m'],
                           c=sample[aqi_col], cmap='RdYlGn_r', s=15, alpha=0.6)
    axes[0,0].set_xlabel('Temperature (°C)')
    axes[0,0].set_ylabel('Humidity (%)')
    axes[0,0].set_title('Temperature vs Humidity (color = AQI)')
    plt.colorbar(sc, ax=axes[0,0], label='AQI')

# 2. Wind Speed vs AQI
if 'wind_speed_10m' in featured.columns and aqi_col in featured.columns:
    axes[0,1].scatter(featured['wind_speed_10m'], featured[aqi_col],
                      alpha=0.4, s=10, color='#00d4ff')
    axes[0,1].set_xlabel('Wind Speed (m/s)')
    axes[0,1].set_ylabel('AQI')
    axes[0,1].set_title('Wind Speed vs AQI')
    axes[0,1].grid(True, alpha=0.2)

# 3. Precipitation vs AQI
if 'precipitation' in featured.columns and aqi_col in featured.columns:
    axes[1,0].scatter(featured['precipitation'], featured[aqi_col],
                      alpha=0.4, s=10, color='#4da6ff')
    axes[1,0].set_xlabel('Precipitation (mm)')
    axes[1,0].set_ylabel('AQI')
    axes[1,0].set_title('Precipitation vs AQI (rain cleans the air)')
    axes[1,0].grid(True, alpha=0.2)

# 4. AQI Lag-1 vs Current AQI
if 'aqi_lag_1' in featured.columns and aqi_col in featured.columns:
    valid = featured[['aqi_lag_1', aqi_col]].dropna()
    axes[1,1].scatter(valid['aqi_lag_1'], valid[aqi_col], alpha=0.4, s=10, color='#ff7e00')
    axes[1,1].set_xlabel('AQI (t-1 hour)')
    axes[1,1].set_ylabel('AQI (current)')
    axes[1,1].set_title(f'AQI Auto-correlation (r={valid["aqi_lag_1"].corr(valid[aqi_col]):.3f})')
    axes[1,1].grid(True, alpha=0.2)

plt.suptitle('Feature Interaction Analysis', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Natural Language Explanation Generator

For every prediction, the system generates a human-readable sentence explaining the result.

In [ ]:
def generate_nl_explanation(feature_impacts, prediction_value):
    """Generate natural-language explanation from feature impacts."""
    from utils.aqi_utils import classify_aqi

    category = classify_aqi(prediction_value).value

    positive = [d for d in feature_impacts if d.get('direction') == 'positive'][:2]
    negative = [d for d in feature_impacts if d.get('direction') == 'negative'][:2]

    sentences = [
        f'The model predicts an AQI of {prediction_value:.0f} ({category}) for Hyderabad.'
    ]

    if positive:
        names = [d['feature'].replace('_', ' ') for d in positive]
        sentences.append(f'Key factors increasing AQI: {", ".join(names)}.')

    if negative:
        names = [d['feature'].replace('_', ' ') for d in negative]
        sentences.append(f'Key factors decreasing AQI: {", ".join(names)}.')

    if prediction_value >= 200:
        sentences.append('⚠ ALERT: Air quality is at hazardous levels. Stay indoors and limit outdoor activity.')
    elif prediction_value >= 150:
        sentences.append('⚠ Air quality is unhealthy. Sensitive groups should limit outdoor activity.')
    elif prediction_value <= 50:
        sentences.append('Air quality is good. Enjoy outdoor activities.')

    return ' '.join(sentences)

# Example
sample_drivers = [
    {'feature': 'aqi_lag_1', 'direction': 'positive', 'importance': 0.35},
    {'feature': 'relative_humidity_2m', 'direction': 'positive', 'importance': 0.18},
    {'feature': 'wind_speed_10m', 'direction': 'negative', 'importance': 0.12},
    {'feature': 'precipitation', 'direction': 'negative', 'importance': 0.08},
]

for pred_val in [45, 95, 165, 250]:
    print(f'\nPrediction: {pred_val}')
    print(generate_nl_explanation(sample_drivers, pred_val))

---
## Summary

| Method | Status | When Used |
|--------|--------|-----------|
| **SHAP TreeExplainer** | Primary | When best model is tree-based (RF, XGBoost, LightGBM) |
| **SHAP KernelExplainer** | Fallback | For Ridge or other non-tree models |
| **Correlation** | Always available | Pearson r between each feature and AQI |
| **Natural Language** | Always available | Template-based NL from top drivers |

**Key insight:** SHAP provides per-prediction explanations — it tells you *why this specific forecast* was made, not just what features matter on average. This is critical for trust and debugging.

**Next:** Notebook 05 — Inference, Alerts & Forecast Visualization